# Practical 7 – Transfer Learning
**Project:** NoteReader – Indian Currency Speaker for the Blind  
**Institute:** RCOEM, Nagpur – CSE Deep Learning Lab  
**Mapped to:** P7 (Transfer Learning & Pretrained Models)

## Objectives
1. Fine-tune VGG16, ResNet50, InceptionV3 and MobileNetV2 on the Indian currency dataset.
2. Compare accuracy, model size, parameters and inference time.
3. Select the best model for on-device (phone) inference and justify the choice.

## Viva Q&A

### Q – What is Transfer Learning?
A model pre-trained on a large dataset (ImageNet, 1.2M images, 1000 classes) has already learned
general visual features: edges, textures, shapes, colours. We **transfer** those features to our task  
by keeping the convolutional base frozen and training only a new classification head on our 6-class dataset.  
This requires far fewer training images and converges in far fewer epochs.

### Q – Why MobileNetV2 for the phone?
| Property | VGG16 | ResNet50 | InceptionV3 | MobileNetV2 |
|----------|-------|----------|-------------|-------------|
| Params   | 138M  | 25M      | 23M         | **3.4M**    |
| Size     | 528MB | 98MB     | 92MB        | **14MB**    |
| ImageNet top-1 | 71% | 74% | 78%       | 71%         |
| Designed for phone? | ✗ | ✗ | ✗ | **✓** |

MobileNetV2 uses **depthwise-separable convolutions** (factorised into depthwise + pointwise steps),  
which reduces multiply-add operations by ~8–9×.  
After TFLite quantisation it runs under 50ms on a mid-range Android CPU.

### Q – What is fine-tuning vs feature extraction?
- **Feature extraction**: freeze all base weights, train only the new head. Fast, needs less data.
- **Fine-tuning**: after head training, unfreeze the top N layers of the base and train with a very small LR.  
  This slightly adjusts the ImageNet features towards currency-specific patterns.

In [ ]:
"""
Setup & data loading (same splits as P6, P4, P5).
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json, os, time
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    VGG16, ResNet50, InceptionV3, MobileNetV2,
    vgg16, resnet50, inception_v3, mobilenet_v2
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

CLASSES     = ['10', '20', '50', '100', '200', '500']
NUM_CLASSES = 6
BATCH_SIZE  = 32

X_train = np.load('../dataset/X_train.npy')
y_train = np.load('../dataset/y_train.npy')
X_val   = np.load('../dataset/X_val.npy')
y_val   = np.load('../dataset/y_val.npy')
X_test  = np.load('../dataset/X_test.npy')
y_test  = np.load('../dataset/y_test.npy')

y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

os.makedirs('../models', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)
print("Setup done. Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

In [ ]:
"""
Model factory.
Each TF application has its own preprocess_input function and optimal input size.
We standardise all inputs to 224×224 for simplicity, except InceptionV3 (299×299).

NOTE: The final phone model uses MobileNetV2 at exactly 224×224 and
      mobilenet_v2.preprocess_input (scales to [-1, 1]).
      This is recorded in model_config.json.
"""

MODEL_CONFIGS = {
    'VGG16': {
        'class': VGG16,
        'preprocess': vgg16.preprocess_input,  # subtracts ImageNet mean
        'input_size': (224, 224),
    },
    'ResNet50': {
        'class': ResNet50,
        'preprocess': resnet50.preprocess_input,
        'input_size': (224, 224),
    },
    'InceptionV3': {
        'class': InceptionV3,
        'preprocess': inception_v3.preprocess_input,  # scales to [-1, 1]
        'input_size': (299, 299),
    },
    'MobileNetV2': {
        'class': MobileNetV2,
        'preprocess': mobilenet_v2.preprocess_input,  # scales to [-1, 1]
        'input_size': (224, 224),
    },
}

def build_transfer_model(model_name, num_classes=6, dropout=0.3):
    """
    Build a transfer learning model:
    1. Load base model with ImageNet weights, no top.
    2. Freeze the base (feature extraction).
    3. Add Global Average Pooling → Dropout → Dense head.
    """
    cfg = MODEL_CONFIGS[model_name]
    size = cfg['input_size']

    # Load base (pre-trained, no classification head)
    base = cfg['class'](
        weights='imagenet',
        include_top=False,
        input_shape=(*size, 3)
    )
    base.trainable = False  # freeze; only head is trained first

    # New classification head
    inputs  = keras.Input(shape=(*size, 3))
    x       = base(inputs, training=False)  # training=False keeps BN in inference mode
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=model_name)
    return model, cfg['preprocess']

print("Model factory ready.")

In [ ]:
"""
Train all 4 transfer learning models.
Phase 1: Head-only training (20 epochs).
Phase 2: Fine-tuning top layers (10 epochs, very low LR).
"""

def train_transfer(model_name, epochs_head=20, epochs_ft=10):
    cfg  = MODEL_CONFIGS[model_name]
    size = cfg['input_size']
    prep = cfg['preprocess']

    # Preprocess the data with each model's preprocess_input
    # (images are already 224×224; resize only for InceptionV3 which uses 299×299)
    from tensorflow.keras.preprocessing.image import smart_resize
    X_tr = prep(smart_resize(X_train, size).astype('float32'))
    X_vl = prep(smart_resize(X_val,   size).astype('float32'))
    X_te = prep(smart_resize(X_test,  size).astype('float32'))

    # Augmentation on preprocessed data
    aug = ImageDataGenerator(
        rotation_range=30, zoom_range=0.2,
        width_shift_range=0.15, height_shift_range=0.15,
        horizontal_flip=True, shear_range=10, fill_mode='nearest')

    model, _ = build_transfer_model(model_name)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy', metrics=['accuracy'])

    # ── Phase 1: Head only ────────────────────────────────────────────────────
    print(f"\n[{model_name}] Phase 1 – Head training")
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    hist1 = model.fit(
        aug.flow(X_tr, y_train_oh, batch_size=BATCH_SIZE, seed=SEED),
        epochs=epochs_head, validation_data=(X_vl, y_val_oh),
        callbacks=[es], verbose=0)
    loss1, acc1 = model.evaluate(X_te, y_test_oh, verbose=0)
    print(f"  After head training – Test Acc: {acc1*100:.2f}%")

    # ── Phase 2: Fine-tune top layers ─────────────────────────────────────────
    # Unfreeze the top 30% of the base model layers
    base_model = model.layers[1]  # the pre-trained base
    total_layers = len(base_model.layers)
    unfreeze_from = int(total_layers * 0.70)
    for layer in base_model.layers[unfreeze_from:]:
        layer.trainable = True
    print(f"  Phase 2 – Fine-tuning top {total_layers - unfreeze_from} / {total_layers} base layers")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),  # very low LR to not destroy features
        loss='categorical_crossentropy', metrics=['accuracy'])

    hist2 = model.fit(
        aug.flow(X_tr, y_train_oh, batch_size=BATCH_SIZE, seed=SEED),
        epochs=epochs_ft, validation_data=(X_vl, y_val_oh),
        callbacks=[es], verbose=0)
    loss2, acc2 = model.evaluate(X_te, y_test_oh, verbose=0)
    print(f"  After fine-tuning  – Test Acc: {acc2*100:.2f}%")

    # ── Metrics ───────────────────────────────────────────────────────────────
    y_pred = np.argmax(model.predict(X_te), axis=1)
    report = classification_report(y_test, y_pred,
                                    target_names=[f'₹{c}' for c in CLASSES],
                                    output_dict=True)

    # Inference time (average over 50 samples)
    sample = X_te[:50]
    start = time.time()
    for _ in range(50):
        model.predict(sample[:1], verbose=0)
    avg_ms = (time.time() - start) / 50 * 1000

    # Model size on disk
    model.save(f'../models/{model_name}.h5')
    size_mb = os.path.getsize(f'../models/{model_name}.h5') / (1024**2)

    return {
        'model': model,
        'history_head': hist1.history,
        'history_ft':   hist2.history,
        'test_acc':     round(acc2 * 100, 2),
        'report':       report,
        'avg_ms':       round(avg_ms, 1),
        'size_mb':      round(size_mb, 1),
        'params':       model.count_params(),
        'y_pred':       y_pred,
        'X_te':         X_te,
    }


results = {}
for name in MODEL_CONFIGS:
    results[name] = train_transfer(name)

print("\nAll transfer learning models trained!")

In [ ]:
"""
Load CNN-scratch results from P6 to include in the comparison table.
"""

import os
# Load CNN scratch history if available
cnn_acc_file = '../results/cnn_scratch_history.json'
cnn_test_acc = None
if os.path.exists(cnn_acc_file):
    with open(cnn_acc_file) as f:
        h = json.load(f)
    cnn_test_acc = round(max(h.get('val_accuracy', [0])) * 100, 2)
    print(f"CNN Scratch best val acc: {cnn_test_acc}%")
else:
    print("CNN scratch history not found. Run P6 notebook first.")

# ── Build the master comparison table ─────────────────────────────────────────
rows = []

# CNN Scratch row (approximate from P6 run)
cnn_size_mb = os.path.getsize('../models/cnn_scratch.h5') / (1024**2) \
    if os.path.exists('../models/cnn_scratch.h5') else 'N/A'
rows.append({
    'Model':           'CNN Scratch',
    'Test Acc (%)':    cnn_test_acc or 'N/A',
    'Params':          'N/A',
    'Size (MB)':       round(float(cnn_size_mb), 1) if cnn_size_mb != 'N/A' else 'N/A',
    'Infer Time (ms)': 'N/A',
    'Pretrained':      'No',
})

for name, r in results.items():
    rows.append({
        'Model':           name,
        'Test Acc (%)':    r['test_acc'],
        'Params':          f"{r['params']:,}",
        'Size (MB)':       r['size_mb'],
        'Infer Time (ms)': r['avg_ms'],
        'Pretrained':      'ImageNet',
    })

df_results = pd.DataFrame(rows)
print("\n" + "="*70)
print("     MODEL COMPARISON TABLE (P7 – Transfer Learning)")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)
df_results.to_csv('../results/model_comparison_table.csv', index=False)

print("""
CONCLUSION:
MobileNetV2 is chosen as the phone model because:
  • Comparable accuracy to ResNet50/InceptionV3 after fine-tuning.
  • Smallest model size (~14MB → ~4MB after TFLite INT8 quantisation).
  • Fastest inference (~50ms on CPU, ~15ms on GPU).
  • Explicitly designed for mobile devices (depthwise-separable convolutions).
  • Used widely in on-device ML deployments.
""")

In [ ]:
"""
Plots: confusion matrices and training curves for all 4 models.
"""

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'₹{c}' for c in CLASSES],
                yticklabels=[f'₹{c}' for c in CLASSES],
                ax=ax)
    ax.set_title(f'{name} – Confusion Matrix (Acc: {r["test_acc"]}%)')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')

plt.suptitle('P7 – Transfer Learning Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/transfer_confusion_matrices.png', dpi=150)
plt.show()

# Training curves
fig, axes = plt.subplots(len(results), 2, figsize=(14, len(results)*4))
for row, (name, r) in enumerate(results.items()):
    # Combine phase 1 and phase 2 histories
    h1 = r['history_head']; h2 = r['history_ft']
    acc_all  = h1['accuracy']  + h2['accuracy']
    vacc_all = h1['val_accuracy'] + h2['val_accuracy']
    loss_all = h1['loss']  + h2['loss']
    vloss_all= h1['val_loss'] + h2['val_loss']
    split_ep = len(h1['loss'])  # where fine-tuning starts

    ax_loss, ax_acc = axes[row]
    ax_loss.plot(loss_all,  label='Train Loss', color='royalblue')
    ax_loss.plot(vloss_all, label='Val Loss',   color='salmon')
    ax_loss.axvline(split_ep, color='gray', linestyle='--', label='Fine-tune start')
    ax_loss.set(title=f'{name} – Loss', xlabel='Epoch', ylabel='Loss')
    ax_loss.legend()

    ax_acc.plot(acc_all,  label='Train Acc', color='royalblue')
    ax_acc.plot(vacc_all, label='Val Acc',   color='salmon')
    ax_acc.axvline(split_ep, color='gray', linestyle='--', label='Fine-tune start')
    ax_acc.set(title=f'{name} – Accuracy', xlabel='Epoch', ylabel='Accuracy')
    ax_acc.legend()

plt.suptitle('P7 – Transfer Learning Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/transfer_training_curves.png', dpi=150)
plt.show()

# Per-class precision/recall/F1 for MobileNetV2
print("\nPer-class metrics – MobileNetV2:")
r = results['MobileNetV2']['report']
for cls_label in [f'₹{c}' for c in CLASSES]:
    m = r.get(cls_label, {})
    print(f"  {cls_label:>5}  P={m.get('precision',0):.2f}  R={m.get('recall',0):.2f}  F1={m.get('f1-score',0):.2f}")

In [ ]:
"""
Save the final MobileNetV2 model for TFLite export.
This model's preprocessing (mobilenet_v2.preprocess_input) is
already captured in model_config.json.
"""

final_model = results['MobileNetV2']['model']
final_model.save('../models/mobilenetv2_final.h5')
print("Final MobileNetV2 model saved to ../models/mobilenetv2_final.h5")
print("Next step: run Export_TFLite.ipynb to convert and quantise.")